# Phase 3A: Model Training & Walk-Forward Validation

This notebook trains Ridge and LightGBM regressors on the long-format NIFTY-50 modeling dataset using **purged walk-forward cross-validation**.

## Execution timing contract

| Stage | Timing |
|-------|--------|
| **Signal** | Market close on day *t* — features at row *t* use all information available at close *t* (no feature shift) |
| **Execution** | Market open on day *t+1* — enforced in the Phase 3B backtesting engine |

Targets are close-to-close forward returns from \(P_t\) to \(P_{t+k}\) for \(k \in \{5, 21\}\).

In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("..")

from src.config import SYMBOLS
from src.models import (
    EXECUTION_TIMING,
    build_modeling_dataset,
    run_walk_forward_cv,
    purged_walk_forward_splits,
    get_backtest_execution_policy,
)

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "ggplot")
plt.rcParams["figure.figsize"] = (12, 6)

## 1. Execution Policy
Document the signal and execution timing contract consumed by the Phase 3B backtesting engine.

In [ ]:
policy = get_backtest_execution_policy()
print("=== BACKTEST EXECUTION POLICY ===")
for key, value in policy.items():
    print(f"  {key}: {value}")

## 2. Build Long-Format Modeling Dataset
Convert the wide feature matrix into one row per (Date, Symbol) and add T+5 / T+21 forward return targets.

In [ ]:
# Use a representative subset for interactive runs; set to SYMBOLS for full universe
train_symbols = ['INFY', 'TCS', 'RELIANCE', 'HDFCBANK', 'SBIN', 'M&M']

dataset, feature_cols = build_modeling_dataset(
    symbols=train_symbols,
    start_date="2010-01-01",
    end_date="2021-04-30",
)

print(f"Dataset shape: {dataset.shape}")
print(f"Base feature count: {len(feature_cols)}")
print(f"Date range: {dataset['Date'].min().date()} to {dataset['Date'].max().date()}")
print(f"\nTarget summary (fwd_ret_5):")
print(dataset['fwd_ret_5'].describe())
dataset.head()

## 3. Purged Walk-Forward Splits
Inspect fold boundaries and purge gaps.

In [ ]:
splits = list(purged_walk_forward_splits(dataset['Date']))
print(f"Number of folds: {len(splits)}")

for i, (train_d, test_d) in enumerate(splits[:3]):
    gap_days = (test_d.min() - train_d.max()).days
    print(f"Fold {i}: train {train_d.min().date()} -> {train_d.max().date()} "
          f"| test {test_d.min().date()} -> {test_d.max().date()} | gap={gap_days}d")

## 4. Walk-Forward Training (Ridge + LightGBM)
Train both models per fold and horizon; collect out-of-sample predictions.

In [ ]:
wf_result = run_walk_forward_cv(dataset=dataset, feature_cols=feature_cols)

print("=== SUMMARY METRICS (OOS mean across folds) ===")
display(wf_result.summary_metrics)

print(f"\nOOS prediction rows: {len(wf_result.oos_predictions)}")
wf_result.oos_predictions.head()

## 5. Metric Visualization
Compare MAE, RMSE, and directional accuracy across models and horizons.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, metric in zip(axes, ['mae', 'rmse', 'directional_accuracy']):
    pivot = wf_result.fold_metrics.pivot_table(
        index='horizon', columns='model', values=metric, aggfunc='mean'
    )
    pivot.plot(kind='bar', ax=ax, rot=0)
    ax.set_title(metric.replace('_', ' ').title())
    ax.set_xlabel('Horizon (days)')

plt.tight_layout()
plt.show()

## 6. Prediction vs Actual (LightGBM, T+21)
Scatter of OOS predictions against realized forward returns.

In [ ]:
oos_21 = wf_result.oos_predictions[wf_result.oos_predictions['horizon'] == 21]

plt.figure(figsize=(8, 6))
plt.scatter(oos_21['actual'], oos_21['lgbm_pred'], alpha=0.15, s=8)
plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
plt.axvline(0, color='gray', linestyle='--', linewidth=0.8)
plt.xlabel('Actual T+21 Return')
plt.ylabel('LightGBM Predicted Return')
plt.title('OOS Predictions vs Actual (T+21)')
plt.show()